> ⚠️ **NOTEBOOK SUPERSEDED.** Ce notebook utilise l'ancien fichier Scan_Tot.xlsx (sans dates, taxonomie obsolète : rest/Ruminating). La version correcte, basée sur le fichier officiel daté Scan_Tot_newVersion_SMN.xlsx, est le **notebook 11**. Conservé pour traçabilité.

---

# Notebook 09 — Objectif 2 : Comportement × environnement d'exercice (Summer 2019)

**Complément au notebook 08.** Le notebook 08 a relié l'**activité IceTag** aux conditions
**thermiques** (HOBO/THI). Ici on relie la **composition comportementale** (observations de scans)
à l'**environnement d'exercice** (taille de paddock, durée d'accès).

**Pourquoi cet angle :** les scans comportementaux n'ont pas de timestamp précis exploitable, ce
qui empêche une synchronisation fiable avec la météo au bin près (limite documentée). En revanche,
chaque observation est associée à une **condition expérimentale** (`Trt` = taille_paddock - durée),
qui est une variable d'environnement contrôlée. On peut donc analyser proprement comment
l'environnement d'exercice influence le comportement, sans aucune hypothèse de date.

**Source :** `Scan_Tot.xlsx` (composition comportementale agrégée, % par catégorie).

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
from scipy.stats import spearmanr, kruskal
import warnings
warnings.filterwarnings('ignore')

PROJECT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
OUT = PROJECT / 'reports' / 'objective2_environnement'
OUT.mkdir(parents=True, exist_ok=True)
SCAN_TOT = PROJECT / 'Données completes' / 'Scan_Tot.xlsx'
print('OK' if SCAN_TOT.exists() else 'MANQUANT')

OK


## 1. Chargement et décodage de l'environnement d'exercice

In [2]:
st = pd.read_excel(SCAN_TOT)
summer = st[st['Experiment'] == 'Summer2019'].copy()

def parse_trt(t):
    m = re.match(r'\s*(\d+)\s*-\s*(\d+)\s*h', str(t))
    return (int(m.group(1)), int(m.group(2))) if m else (np.nan, np.nan)
summer[['pad_size', 'duration_h']] = summer['Trt'].apply(lambda t: pd.Series(parse_trt(t)))
summer = summer.dropna(subset=['pad_size', 'duration_h'])

behav_cols = ['Pct_locomotion', 'Pct_rest', 'Pct_lying', 'Pct_Explo',
              'Pct_eating', 'Pct_Ruminating', 'Pct_Social', 'Pct_Maintenance']
print(f"Observations : {len(summer)}")
print(f"Tailles de paddock : {sorted(summer['pad_size'].unique())} m²")
print(f"Durées d'accès : {sorted(summer['duration_h'].unique())} h")
print("\nComposition comportementale moyenne :")
print((summer[behav_cols].mean() * 100).round(1).to_string())

Observations : 58
Tailles de paddock : [np.int64(20), np.int64(40), np.int64(60), np.int64(80)] m²
Durées d'accès : [np.int64(1), np.int64(2)] h

Composition comportementale moyenne :
Pct_locomotion      2.2
Pct_rest           48.5
Pct_lying           0.0
Pct_Explo          13.1
Pct_eating          9.8
Pct_Ruminating     19.5
Pct_Social          3.5
Pct_Maintenance     1.6


## 2. Comportement vs taille de paddock

Hypothèse : plus d'espace → plus de locomotion et d'exploration.

In [3]:
print('=== Composition comportementale (%) par taille de paddock ===')
by_pad = summer.groupby('pad_size')[behav_cols].mean() * 100
print(by_pad.round(1).to_string())
print()
print('=== Corrélation (Spearman) taille paddock vs chaque comportement ===')
rows = []
for c in behav_cols:
    rho, p = spearmanr(summer['pad_size'], summer[c])
    rows.append({'comportement': c.replace('Pct_', ''), 'rho_vs_pad_size': round(rho, 3),
                 'p': round(p, 4), 'signif': 'OUI' if p < 0.05 else 'non'})
res_pad = pd.DataFrame(rows).sort_values('rho_vs_pad_size', ascending=False)
print(res_pad.to_string(index=False))
res_pad.to_csv(OUT / 'summer2019_comportement_vs_paddock.csv', index=False)

=== Composition comportementale (%) par taille de paddock ===
          Pct_locomotion  Pct_rest  Pct_lying  Pct_Explo  Pct_eating  Pct_Ruminating  Pct_Social  Pct_Maintenance
pad_size                                                                                                         
20                   0.3      53.4        0.0       11.4         5.5            24.0         2.3              2.1
40                   0.7      45.2        0.0       11.7        14.6            21.0         3.1              1.9
60                   3.7      46.6        0.0       13.9         7.8            19.7         4.8              1.0
80                   3.8      50.2        0.0       15.2        10.2            13.6         3.6              1.5

=== Corrélation (Spearman) taille paddock vs chaque comportement ===
comportement  rho_vs_pad_size      p signif
  locomotion            0.321 0.0139    OUI
      eating            0.162 0.2235    non
       Explo            0.132 0.3244    non
      So

## 3. Comportement vs durée d'accès

In [4]:
print('=== Composition comportementale (%) par durée d\'accès ===')
by_dur = summer.groupby('duration_h')[behav_cols].mean() * 100
print(by_dur.round(1).to_string())
print()
print('=== Test (Kruskal-Wallis) durée 1h vs 2h par comportement ===')
rows = []
for c in behav_cols:
    g1 = summer[summer['duration_h'] == 1][c].dropna()
    g2 = summer[summer['duration_h'] == 2][c].dropna()
    try:
        stat, p = kruskal(g1, g2)
    except Exception:
        p = np.nan
    rows.append({'comportement': c.replace('Pct_', ''), 'moy_1h': round(g1.mean()*100, 1),
                 'moy_2h': round(g2.mean()*100, 1), 'p': round(p, 4) if not np.isnan(p) else None,
                 'signif': 'OUI' if (not np.isnan(p) and p < 0.05) else 'non'})
res_dur = pd.DataFrame(rows)
print(res_dur.to_string(index=False))
res_dur.to_csv(OUT / 'summer2019_comportement_vs_duree.csv', index=False)

=== Composition comportementale (%) par durée d'accès ===
            Pct_locomotion  Pct_rest  Pct_lying  Pct_Explo  Pct_eating  Pct_Ruminating  Pct_Social  Pct_Maintenance
duration_h                                                                                                         
1                      1.1      42.6        0.0       12.7         9.5            28.4         3.7              1.0
2                      3.2      54.5        0.0       13.5        10.1            10.5         3.3              2.2

=== Test (Kruskal-Wallis) durée 1h vs 2h par comportement ===
comportement  moy_1h  moy_2h      p signif
  locomotion     1.1     3.2 0.0083    OUI
        rest    42.6    54.5 0.0733    non
       lying     0.0     0.0    NaN    non
       Explo    12.7    13.5 0.5519    non
      eating     9.5    10.1 0.2405    non
  Ruminating    28.4    10.5 0.3757    non
      Social     3.7     3.3 0.2181    non
 Maintenance     1.0     2.2 0.0437    OUI


## 4. Visualisation

In [5]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

main = ['Pct_locomotion', 'Pct_rest', 'Pct_Explo', 'Pct_eating', 'Pct_Ruminating']
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

by_pad[main].plot(kind='bar', ax=axes[0])
axes[0].set_title('Composition comportementale par taille de paddock')
axes[0].set_xlabel('Taille paddock (m²)'); axes[0].set_ylabel('% du temps')
axes[0].legend(fontsize=8, labels=[c.replace('Pct_', '') for c in main])

by_dur[main].plot(kind='bar', ax=axes[1])
axes[1].set_title("Composition comportementale par durée d'accès")
axes[1].set_xlabel("Durée (h)"); axes[1].set_ylabel('% du temps')
axes[1].legend(fontsize=8, labels=[c.replace('Pct_', '') for c in main])

fig.suptitle('Summer 2019 — Comportement × environnement d\'exercice', fontsize=13)
fig.tight_layout()
fig.savefig(OUT / 'summer2019_comportement_environnement.png', dpi=120, bbox_inches='tight')
print('Figure sauvegardée :', OUT / 'summer2019_comportement_environnement.png')
plt.show()

Figure sauvegardée : /Users/alioubarry/PROJECT/mcgill_iot_cattle/reports/objective2_environnement/summer2019_comportement_environnement.png


## 5. Synthèse

In [6]:
lines = []
lines.append('# Objectif 2 — Comportement × environnement d\'exercice (Summer 2019)\n')
lines.append(f'Observations analysées : {len(summer)} (Scan_Tot, composition comportementale).\n')
lines.append('## Comportement vs taille de paddock')
lines.append(res_pad.to_string(index=False))
lines.append('')
sig_pad = res_pad[res_pad['signif'] == 'OUI']
if len(sig_pad):
    for _, r in sig_pad.iterrows():
        sens = 'augmente' if r['rho_vs_pad_size'] > 0 else 'diminue'
        lines.append(f"- {r['comportement']} {sens} significativement avec la taille du paddock (rho={r['rho_vs_pad_size']}).")
else:
    lines.append('- Aucun comportement ne varie significativement avec la taille du paddock.')
lines.append('\n## Comportement vs durée d\'accès')
lines.append(res_dur.to_string(index=False))
lines.append('')
lines.append('## Limite documentée')
lines.append('Les scans comportementaux ne disposent pas de timestamp précis exploitable. La '
             'synchronisation fine scan-météo (au bin de 15 min) n\'est donc pas possible de façon '
             'fiable. L\'analyse environnement-comportement est conduite via la condition '
             'expérimentale (taille de paddock, durée), variable d\'environnement contrôlée et datée '
             'sans ambiguïté. Recommandation : horodater les observations comportementales pour '
             'permettre une synchronisation directe avec les capteurs environnementaux.')
note = '\n'.join(lines)
(OUT / 'objectif2_comportement_synthese.md').write_text(note, encoding='utf-8')
print(note)

# Objectif 2 — Comportement × environnement d'exercice (Summer 2019)

Observations analysées : 58 (Scan_Tot, composition comportementale).

## Comportement vs taille de paddock
comportement  rho_vs_pad_size      p signif
  locomotion            0.321 0.0139    OUI
      eating            0.162 0.2235    non
       Explo            0.132 0.3244    non
      Social            0.064 0.6353    non
        rest           -0.036 0.7892    non
  Ruminating           -0.129 0.3342    non
 Maintenance           -0.144 0.2792    non
       lying              NaN    NaN    non

- locomotion augmente significativement avec la taille du paddock (rho=0.321).

## Comportement vs durée d'accès
comportement  moy_1h  moy_2h      p signif
  locomotion     1.1     3.2 0.0083    OUI
        rest    42.6    54.5 0.0733    non
       lying     0.0     0.0    NaN    non
       Explo    12.7    13.5 0.5519    non
      eating     9.5    10.1 0.2405    non
  Ruminating    28.4    10.5 0.3757    non
      Social